This part of the script was executed in Google colab due to ressoruce restraints. The files are the same ones generated in the rest of the repo. Google Drive link: https://drive.google.com/drive/folders/1hID0exGhiKZbEKDbP0J2KO9Oqcaqm-x3?usp=sharing

In [1]:
# Install miniconda
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!chmod +x miniconda.sh
!./miniconda.sh -b -f -p /usr/local
!conda update -n base -c defaults conda -y

# Environment with python 3.11 for compatability
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda create -n myenv python=3.11 -y

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local
Jupyter detected...

CondaToSNonInteractiveError: Terms of Service have not been accepted for the following channels. Please accept or remove them before proceeding:
    - https://repo.anaconda.com/pkgs/main
    - https://repo.anaconda.com/pkgs/r

To accept these channels' Terms of Service, run the following commands:
    conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    conda tos accept --override-channels --channel https:/

In [2]:
%%shell
eval "$(conda shell.bash hook)"
conda activate myenv
pip install -U scgpt "torch<=2.2.2" "numpy<2" "umap-learn<0.5.7"
pip install gdown pooch
pip install -q ipykernel
python -m ipykernel install --user --name myenv --display-name "Python 3.11 (myenv)"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of ml-dtypes to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of zarr to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.i

In [3]:
import os
from google.colab import drive
drive.mount('/content/drive')

print(os.listdir('/content/drive/MyDrive/Colab Notebooks'))

Mounted at /content/drive
['scgpt.ipynb', 'scGPT_human', 'adata_integrated.h5ad', 'scGPT_CT', 'scgpt2.ipynb']


In [6]:
import os

model_dir = "/content/drive/MyDrive/Colab Notebooks/scGPT_CT"
old_path = os.path.join(model_dir, "model.pt")
new_path = os.path.join(model_dir, "best_model.pt")

if os.path.exists(new_path):
    print("best_model.pt already exists, nothing to do.")
elif os.path.exists(old_path):
    os.rename(old_path, new_path)
    print("Renamed model.pt -> best_model.pt")
else:
    print("Neither model.pt nor best_model.pt found — check the path.")

Renamed model.pt -> best_model.pt


In [7]:
%%writefile run_analysis.py
import os
from pathlib import Path

import gdown
import pooch
import torch
import scanpy as sc
import anndata as ad
import scgpt as scg

# Check GPU availability
print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

# Path to pretrained scGPT model weights
model_dir = Path("/content/drive/MyDrive/Colab Notebooks/scGPT_CT")

# Load input AnnData
sample_data_path = '/content/drive/MyDrive/Colab Notebooks/adata_integrated.h5ad'
adata = sc.read_h5ad(sample_data_path)

# scGPT expects gene symbols in a specific .var column
gene_col = "Gene Symbol"
adata.var[gene_col] = adata.var.index.values

# Compute cell embeddings using scGPT
print(f"Starte Embedding für {adata.n_obs} Zellen...")
embed_adata = scg.tasks.embed_data(adata, model_dir, gene_col=gene_col, batch_size=64)

# Save output AnnData containing embeddings
ad.settings.allow_write_nullable_strings = True
output_path = '/content/drive/MyDrive/Colab Notebooks/adata_embedded.h5ad'
embed_adata.write_h5ad(output_path)
print(f"Saved under: {output_path}")

Overwriting run_analysis.py


In [8]:
%%shell
eval "$(conda shell.bash hook)"
conda activate myenv
python -u run_analysis.py

/usr/local/envs/myenv/lib/python3.11/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/usr/local/envs/myenv/lib/python3.11/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
PyTorch version: 2.2.2+cu121
CUDA available : True
Starte Embedding für 49444 Zellen...
scGPT - INFO - match 1916/2000 genes in vocabulary of size 60697.
/usr/local/envs/myenv/lib/python3.11/site-packages/scgpt/model/model.py:77: UserWarning: flash-attn is not installed, using pytorch transformer instead. Set use_fast_transformer=False to avoid this warning. Installing flash-attn is highly recommended.
  warnings.warn(

/usr/local/envs/myenv/lib/python3.11/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings
Saved under: /

In [10]:
%%shell
eval "$(conda shell.bash hook)"
conda activate myenv
python -c "
import scanpy as sc
import numpy as np

adata = sc.read_h5ad('/content/drive/MyDrive/Colab Notebooks/adata_embedded.h5ad')
print(adata)
print()
print('obsm keys:', list(adata.obsm.keys()))

emb = adata.obsm.get('X_scGPT')
if emb is not None:
    print('Shape:', emb.shape)
    print('contains NaN:', np.isnan(emb).any())
    print('all zero:', np.all(emb == 0))
    print('first values:', emb[0, :5])
"

AnnData object with n_obs × n_vars = 49444 × 1916
    obs: 'library_uuid', 'assay_ontology_term_id', 'mapped_reference_annotation', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'cell_state', 'sample_uuid', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'disease_state', 'suspension_enriched_cell_types', 'suspension_uuid', 'suspension_type', 'donor_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'sex_ontology_term_id', 'Processing_Cohort', 'ct_cov', 'ind_cov', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', '_scvi_batch', '_scvi_labels'
    var: 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'ensembl_id', 'mt', 'ribo', 'hb', 'n_cells_by_counts'